# 第11章: 多層ニューラルネットワークをゼロから実装する

この Notebook は、原本 `machine-learning-book/ch11/ch11.ipynb` を最新の Python 環境と
`pytest --nbmake` による CI 実行向けに移行したものです。

原本の教育意図を保ちながら、次の点を更新しています。

- ネットワーク経由の `fetch_openml('mnist_784')` は使わず、ローカルで利用できる `load_digits()` に置き換える
- MLP の順伝播、逆伝播、ミニバッチ学習ループは NumPy 実装のまま残し、章の学習目的を維持する
- 学習エポック数とデータサイズを CI で現実的な時間に収まる構成へ調整する


## この Notebook で確認すること

- 原本図版を `src/` 配下から参照できることを確認する
- シグモイド関数と多層パーセプトロンの順伝播を NumPy で再現する
- `load_digits()` データセットを前処理し、訓練・検証・テストに分割する
- 自前実装の MLP をミニバッチ学習で訓練し、損失と精度の推移を確認する
- テストセット精度と誤分類例を確認し、章の内容を継続検証可能な形で残す


In [ ]:
from importlib.metadata import version
from pathlib import Path
import platform
import sys

from IPython.display import Image, display
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "machine-learning-book").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("リポジトリルートを見つけられませんでした。")


REPO_ROOT = find_repo_root()
CHAPTER_DIR = REPO_ROOT / "machine-learning-book" / "ch11"
FIGURE_DIR = CHAPTER_DIR / "figures"

assert FIGURE_DIR.exists(), f"図版ディレクトリが見つかりません: {FIGURE_DIR}"

print(f"Python 実行ファイル: {sys.executable}")
print(f"Python バージョン: {platform.python_version()}")
print(f"Matplotlib バックエンド: {matplotlib.get_backend()}")
print(f"Chapter directory: {CHAPTER_DIR}")


In [ ]:
package_versions = pd.DataFrame(
    [
        ("numpy", version("numpy")),
        ("pandas", version("pandas")),
        ("matplotlib", version("matplotlib")),
        ("scikit-learn", version("scikit-learn")),
        ("pytest", version("pytest")),
    ],
    columns=["パッケージ", "バージョン"],
)
package_versions


## 原本図版の参照

単層ネットワークと多層ネットワークの図を読み込み、移行版 Notebook から原本アセットに安定してアクセスできることを確認します。


In [ ]:
for figure_name in ["11_01.png", "11_02.png", "11_03.png"]:
    print(figure_name)
    display(Image(filename=str(FIGURE_DIR / figure_name), width=560))


## シグモイド関数と順伝播の確認

原本の前半で扱う活性化関数と順伝播の考え方を、小さな NumPy 配列で確認します。


In [ ]:
def sigmoid(z: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-z))


sample_input = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
sample_hidden_weights = np.array([[0.2, -0.4, 0.1], [0.7, 0.3, -0.5]])
sample_hidden_bias = np.array([0.1, -0.2])
sample_x = np.array([[0.5, -0.3, 0.8]])

hidden_net = np.dot(sample_x, sample_hidden_weights.T) + sample_hidden_bias
hidden_act = sigmoid(hidden_net)

display(pd.DataFrame({"z": sample_input, "sigmoid(z)": np.round(sigmoid(sample_input), 4)}))
pd.DataFrame(hidden_act, columns=["hidden_unit_1", "hidden_unit_2"]).round(4)


## 手書き数字データの準備

原本は MNIST をダウンロードしていましたが、移行版ではネットワーク依存を避けるため
scikit-learn 同梱の `load_digits()` を使います。特徴量は 64 次元、画像サイズは 8x8 です。


In [ ]:
digits = load_digits()
X = digits.data.astype(np.float64)
y = digits.target.astype(np.int64)

# ピクセル値を [-1, 1] に正規化
X = ((X / 16.0) - 0.5) * 2.0

X_temp, X_test, y_temp, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=123,
    stratify=y,
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_temp,
    y_temp,
    test_size=0.2,
    random_state=123,
    stratify=y_temp,
)

summary = pd.Series(
    {
        "all_samples": len(X),
        "num_features": X.shape[1],
        "num_classes": len(np.unique(y)),
        "train_size": len(X_train),
        "valid_size": len(X_valid),
        "test_size": len(X_test),
    }
)

display(summary.to_frame(name="値"))
display(pd.Series(y).value_counts().sort_index().to_frame(name="count"))


In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(6.0, 3.0), sharex=True, sharey=True)
axes = axes.flatten()
for digit in range(10):
    axes[digit].imshow(X[y == digit][0].reshape(8, 8), cmap="Greys")
    axes[digit].set_title(str(digit))
    axes[digit].set_xticks([])
    axes[digit].set_yticks([])
plt.tight_layout()
plt.show()
plt.close(fig)


## 多層パーセプトロンの実装

原本の実装方針を引き継ぎ、1 隠れ層の MLP を NumPy で定義します。損失は平均二乗誤差、
出力活性化はシグモイドのままにして、逆伝播も明示的に実装します。


In [ ]:
def int_to_onehot(y: np.ndarray, num_labels: int) -> np.ndarray:
    onehot = np.zeros((y.shape[0], num_labels))
    onehot[np.arange(y.shape[0]), y] = 1.0
    return onehot


class NeuralNetMLP:
    def __init__(self, num_features: int, num_hidden: int, num_classes: int, random_seed: int = 123):
        self.num_classes = num_classes
        rng = np.random.RandomState(random_seed)
        self.weight_h = rng.normal(loc=0.0, scale=0.1, size=(num_hidden, num_features))
        self.bias_h = np.zeros(num_hidden)
        self.weight_out = rng.normal(loc=0.0, scale=0.1, size=(num_classes, num_hidden))
        self.bias_out = np.zeros(num_classes)

    def forward(self, x: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        z_h = np.dot(x, self.weight_h.T) + self.bias_h
        a_h = sigmoid(z_h)
        z_out = np.dot(a_h, self.weight_out.T) + self.bias_out
        a_out = sigmoid(z_out)
        return a_h, a_out

    def backward(
        self,
        x: np.ndarray,
        a_h: np.ndarray,
        a_out: np.ndarray,
        y: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        y_onehot = int_to_onehot(y, self.num_classes)

        d_loss__d_a_out = 2.0 * (a_out - y_onehot) / y.shape[0]
        d_a_out__d_z_out = a_out * (1.0 - a_out)
        delta_out = d_loss__d_a_out * d_a_out__d_z_out

        d_loss__dw_out = np.dot(delta_out.T, a_h)
        d_loss__db_out = np.sum(delta_out, axis=0)

        d_loss__a_h = np.dot(delta_out, self.weight_out)
        d_a_h__d_z_h = a_h * (1.0 - a_h)
        d_loss__d_w_h = np.dot((d_loss__a_h * d_a_h__d_z_h).T, x)
        d_loss__d_b_h = np.sum(d_loss__a_h * d_a_h__d_z_h, axis=0)

        return d_loss__dw_out, d_loss__db_out, d_loss__d_w_h, d_loss__d_b_h


model = NeuralNetMLP(num_features=64, num_hidden=64, num_classes=10)
a_h_preview, a_out_preview = model.forward(X_train[:3])
pd.Series(
    {
        "hidden_activation_shape": str(a_h_preview.shape),
        "output_activation_shape": str(a_out_preview.shape),
        "output_sum_example_0": round(float(a_out_preview[0].sum()), 4),
    }
).to_frame(name="値")


## ミニバッチ学習ループ

原本の訓練ループを踏襲しつつ、`load_digits()` で現実的な時間に収まるようにエポック数を調整します。
訓練時には各 epoch の訓練 MSE、訓練精度、検証精度を記録します。


In [ ]:
def minibatch_generator(
    X: np.ndarray,
    y: np.ndarray,
    minibatch_size: int,
    shuffle: bool = True,
    seed: int | None = None,
):
    indices = np.arange(X.shape[0])
    if shuffle:
        rng = np.random.RandomState(seed)
        rng.shuffle(indices)
    for start_idx in range(0, indices.shape[0], minibatch_size):
        batch_idx = indices[start_idx : start_idx + minibatch_size]
        yield X[batch_idx], y[batch_idx]


def compute_mse_and_acc(
    nnet: NeuralNetMLP,
    X: np.ndarray,
    y: np.ndarray,
    minibatch_size: int = 64,
):
    mse = 0.0
    correct_pred = 0
    num_examples = 0
    num_batches = 0

    for features, targets in minibatch_generator(X, y, minibatch_size=minibatch_size, shuffle=False):
        _, probas = nnet.forward(features)
        onehot_targets = int_to_onehot(targets, num_labels=nnet.num_classes)
        mse += np.mean((onehot_targets - probas) ** 2)
        predicted_labels = np.argmax(probas, axis=1)
        correct_pred += (predicted_labels == targets).sum()
        num_examples += targets.shape[0]
        num_batches += 1

    return mse / num_batches, correct_pred / num_examples


def train(
    model: NeuralNetMLP,
    X_train: np.ndarray,
    y_train: np.ndarray,
    X_valid: np.ndarray,
    y_valid: np.ndarray,
    num_epochs: int = 60,
    learning_rate: float = 0.1,
    minibatch_size: int = 64,
):
    epoch_loss = []
    epoch_train_acc = []
    epoch_valid_acc = []

    for epoch in range(num_epochs):
        for X_train_mini, y_train_mini in minibatch_generator(
            X_train,
            y_train,
            minibatch_size=minibatch_size,
            shuffle=True,
            seed=epoch,
        ):
            a_h, a_out = model.forward(X_train_mini)
            d_w_out, d_b_out, d_w_h, d_b_h = model.backward(X_train_mini, a_h, a_out, y_train_mini)

            model.weight_h -= learning_rate * d_w_h
            model.bias_h -= learning_rate * d_b_h
            model.weight_out -= learning_rate * d_w_out
            model.bias_out -= learning_rate * d_b_out

        train_mse, train_acc = compute_mse_and_acc(model, X_train, y_train, minibatch_size=minibatch_size)
        valid_mse, valid_acc = compute_mse_and_acc(model, X_valid, y_valid, minibatch_size=minibatch_size)

        epoch_loss.append(train_mse)
        epoch_train_acc.append(train_acc * 100.0)
        epoch_valid_acc.append(valid_acc * 100.0)

    return epoch_loss, epoch_train_acc, epoch_valid_acc


np.random.seed(123)
epoch_loss, epoch_train_acc, epoch_valid_acc = train(
    model,
    X_train,
    y_train,
    X_valid,
    y_valid,
    num_epochs=60,
    learning_rate=0.1,
    minibatch_size=64,
)

pd.DataFrame(
    {
        "epoch": [1, 10, 20, 40, 60],
        "train_mse": [epoch_loss[i - 1] for i in [1, 10, 20, 40, 60]],
        "train_acc": [epoch_train_acc[i - 1] for i in [1, 10, 20, 40, 60]],
        "valid_acc": [epoch_valid_acc[i - 1] for i in [1, 10, 20, 40, 60]],
    }
).round(3)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8.6, 3.2))
axes[0].plot(range(1, len(epoch_loss) + 1), epoch_loss, linewidth=2, color="#C44E52")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Training MSE")
axes[0].set_title("Loss Curve")
axes[0].grid(alpha=0.3)

axes[1].plot(range(1, len(epoch_train_acc) + 1), epoch_train_acc, label="Training", linewidth=2)
axes[1].plot(range(1, len(epoch_valid_acc) + 1), epoch_valid_acc, label="Validation", linewidth=2)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].set_title("Accuracy Curve")
axes[1].legend(loc="lower right")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()
plt.close(fig)


## テストセットでの評価

学習済みモデルをテストセットに適用し、MSE と精度を確認します。あわせて誤分類例もいくつか表示します。


In [ ]:
test_mse, test_acc = compute_mse_and_acc(model, X_test, y_test, minibatch_size=64)
_, test_probas = model.forward(X_test)
test_pred = np.argmax(test_probas, axis=1)

display(
    pd.Series(
        {
            "test_mse": round(float(test_mse), 4),
            "test_accuracy_percent": round(float(test_acc * 100.0), 2),
        }
    ).to_frame(name="値")
)

confusion_preview = pd.crosstab(
    pd.Series(y_test, name="true"),
    pd.Series(test_pred, name="pred"),
)
confusion_preview.iloc[:5, :5]


In [ ]:
misclassified_idx = np.flatnonzero(test_pred != y_test)[:9]

fig, axes = plt.subplots(nrows=3, ncols=3, figsize=(5.4, 5.4), sharex=True, sharey=True)
axes = axes.flatten()
for ax, idx in zip(axes, misclassified_idx):
    ax.imshow(X_test[idx].reshape(8, 8), cmap="Greys")
    ax.set_title(f"t={y_test[idx]}, p={test_pred[idx]}")
    ax.set_xticks([])
    ax.set_yticks([])
for ax in axes[len(misclassified_idx):]:
    ax.axis("off")
plt.tight_layout()
plt.show()
plt.close(fig)


## まとめ

この移行版 Notebook では、第11章の中心となるシグモイド関数、順伝播、逆伝播、
NumPy による多層パーセプトロン実装、ミニバッチ学習、評価までを最新環境向けに再構成しました。

原本の MNIST ダウンロード部分は CI 互換性のため `load_digits()` に置き換えていますが、
「ニューラルネットワークをゼロから実装して学習させる」という章の学習目的は維持しています。
